In [13]:
!pip install ifcopenshell

In [ ]:
import ifcopenshell

In [24]:
# Script finds the start and end elevations of ducts and pipes and writes them to an excel file.
# Extract heights via IfcDistributionPorts, fallback to bounding box center Z if ports are missing.
# Version 0.2 - 25.06.2026 - IFC4x3

import ifcopenshell
import ifcopenshell.util.placement
import ifcopenshell.geom
import pandas as pd
import numpy as np

# --- Configuration --- #
model_path = 'LU32_LU_MD_ME_FACHM.ifc'
level_to_process = "05"

# Initialize geometry settings with correct attribute keys
settings = ifcopenshell.geom.settings()
settings.set("use-world-coords", True)

def get_bbox_center_z(element):
    """Calculates the Z-center of the element's bounding box using global geometry."""
    try:
        # Generate geometry. Vertices will be global due to use-world-coords
        shape = ifcopenshell.geom.create_shape(settings, element)
        verts = shape.geometry.verts

        # Extract every 3rd value starting from index 2 (Z coordinates)
        z_coords = [verts[i] for i in range(2, len(verts), 3)]

        if z_coords:
            return (min(z_coords) + max(z_coords)) / 2.0
    except:
        return None
    return None

def get_global_z(placement):
    z = 0.0
    current = placement
    while current:
        if current.RelativePlacement and current.RelativePlacement.Location:
            coords = current.RelativePlacement.Location.Coordinates
            if len(coords) >= 3: z += float(coords[2])
        current = getattr(current, 'PlacementRelTo', None)
    return z

def process_ifc_model(model, level_name=None):
    print(f"Loading model... ({level_name if level_name else 'Full'})")
    element_types = {'IfcPipeSegment', 'IfcDuctSegment', 'IfcPipeFitting', 'IfcDuctFitting', 'IfcDamper', 'IfcDuctSilencer'}
    candidate_elements = []
    element_to_storey = {}

    # Filter elements by storey relationship
    for rel in model.by_type('IfcRelContainedInSpatialStructure'):
        storey_name = getattr(rel.RelatingStructure, 'Name', 'Unknown')
        if level_name and storey_name != level_name: continue
        for element in rel.RelatedElements:
            if any(element.is_a(t) for t in element_types):
                candidate_elements.append(element)
                element_to_storey[element.GlobalId] = storey_name

    print("Pre-mapping ports...")
    port_map = {}
    for rel in model.by_type('IfcRelNests'):
        if rel.RelatingObject in candidate_elements:
            ports = [obj for obj in rel.RelatedObjects if obj.is_a('IfcDistributionPort')]
            port_map.setdefault(rel.RelatingObject, []).extend(ports)
    for rel in model.by_type('IfcRelConnectsPortToElement'):
        if rel.RelatedElement in candidate_elements:
            port_map.setdefault(rel.RelatedElement, []).append(rel.RelatingPort)

    print(f"Processing {len(candidate_elements)} elements...")
    results = []
    failed_count = 0

    for element in candidate_elements:
        elevations = []
        is_estimated = False

        # 1. Port based extraction
        ports = port_map.get(element, [])
        for port in ports:
            try:
                matrix = ifcopenshell.util.placement.get_abs_placement(port.ObjectPlacement)
                elevations.append(float(matrix[2, 3]) if isinstance(matrix, np.ndarray) else float(matrix[2][3]))
            except:
                val = get_global_z(port.ObjectPlacement)
                if val: elevations.append(val)

        # 2. Geometry based fallback (Estimated)
        if not elevations:
            center_z = get_bbox_center_z(element)
            if center_z is not None:
                elevations = [center_z]
                is_estimated = True

        if elevations:
            elevations.sort()
            results.append({
                'Guid': element.GlobalId,
                'Name': getattr(element, 'Name', 'N/A'),
                'Type': element.is_a(),
                'Storey': element_to_storey.get(element.GlobalId, 'N/A'),
                'Start_Elevation': round(elevations[0], 3),
                'End_Elevation': round(elevations[-1], 3),
                'Is_Estimated': is_estimated
            })
        else:
            failed_count += 1

    df = pd.DataFrame(results)
    print(f"\nSummary for Level {level_name}:")
    if not df.empty:
        print(df['Is_Estimated'].value_counts())
    print(f"Failed completely: {failed_count}")
    return df

# --- Execution ---
model = ifcopenshell.open(model_path)
df_res = process_ifc_model(model, level_name=level_to_process)

if not df_res.empty:
    display(df_res.head())
    fname = f'elevations_{level_to_process}.xlsx'
    df_res.to_excel(fname, index=False)
    print(f"\nSaved results to {fname}")

Loading model... (05)
Pre-mapping ports...
Processing 2728 elements...

Summary for Level 05:
Is_Estimated
False    2159
True      569
Name: count, dtype: int64
Failed completely: 0


,Guid,Name,Type,Storey,Start_Elevation,End_Elevation,Is_Estimated
0,3VT9t6PZ59jPDaAu0FUMJp,Brandschutzklappe rund,IfcDamper,05,485.382,485.382,False
1,0Ega3LZwT7$eNZLiQ5HLgk,Rohrbogen,IfcDuctFitting,05,485.340,485.340,False
2,1qVSF84hzEIQRRStlUCgAq,T-Stück,IfcDuctFitting,05,485.533,485.533,False
3,1cZ5o_yHz5FvrUAiJz7z8T,Rohrbogen,IfcDuctFitting,05,485.340,485.340,False
4,0cpEMCn$bFoAT1vaOXRX8p,T-Stück,IfcDuctFitting,05,485.513,485.513,False



Saved results to elevations_05.xlsx


In [15]:
# Auxiliary script to list the storey names in the model for filtering in the first script above.
temp_model = ifcopenshell.open(model_path)

print(f"Available IfcBuildingStorey names in '{model_path}':")
found_levels = False
for storey in temp_model.by_type('IfcBuildingStorey'):
    name = getattr(storey, 'Name', 'N/A')
    long_name = getattr(storey, 'LongName', 'N/A')
    print(f" - Name: '{name}', LongName: '{long_name}'")
    found_levels = True

if not found_levels:
    print("No IfcBuildingStorey elements found in the model.")
else:
    print("\nUse one of these 'Name' or 'LongName' values for 'level_to_process' in the configuration.")


Available IfcBuildingStorey names in 'LU32_LU_MD_ME_FACHM.ifc':
 - Name: '07', LongName: 'None'
 - Name: '06', LongName: 'None'
 - Name: '05', LongName: 'None'
 - Name: '04', LongName: 'None'
 - Name: '03', LongName: 'None'
 - Name: '02', LongName: 'None'
 - Name: '01', LongName: 'None'
 - Name: '00', LongName: 'None'
 - Name: 'U1', LongName: 'None'
 - Name: 'U2', LongName: 'None'
 - Name: 'U3', LongName: 'None'

Use one of these 'Name' or 'LongName' values for 'level_to_process' in the configuration.
